# OTS-02 resource addendum

This additive notebook preserves the original notebook and adds source-backed implementation controls for volatility trading and variance premium.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['ots02_source_pdf_inventory.csv','ots02_edge_framework.csv','ots02_volatility_laws.csv','ots02_option_payoff_reference.csv','ots02_iv_hv_surface_reference.csv','ots02_variance_premium_reference.csv','ots02_live_trading_controls.csv','ots02_adverse_selection_cost_demo.csv','ots02_iv_surface_demo.csv','ots02_variance_premium_demo.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Source inventory and edge framework

The lecture separates model-driven trades, event-driven trades, adverse selection, and transaction costs.

In [ ]:
pdfs = data['ots02_source_pdf_inventory.csv']
edge = data['ots02_edge_framework.csv']
print(pdfs[['file','pages','keyword_hits']].to_string(index=False))
print(edge[['topic','implementation_rule']].to_string(index=False))
assert edge['topic'].str.contains('Adverse selection').any()
assert edge['topic'].str.contains('Costs').any()

## 2. Volatility laws and option payoffs

The lecture's volatility laws are useful only after separating payoff from P&L and direction from volatility exposure.

In [ ]:
laws = data['ots02_volatility_laws.csv']
payoffs = data['ots02_option_payoff_reference.csv']
print(laws.to_string(index=False))
print(payoffs.to_string(index=False))
assert laws['law'].str.contains('mean-reverting').any()
assert payoffs['instrument'].str.contains('Straddle', case=False).any()

## 3. Historical, realized, implied, smile, term structure, surface

Historical/realized volatility comes from returns. Implied volatility comes from option price and is contract-specific.

In [ ]:
surface_ref = data['ots02_iv_hv_surface_reference.csv']
surface = data['ots02_iv_surface_demo.csv']
print(surface_ref.to_string(index=False))
print(surface.head(10).to_string(index=False))
assert surface_ref['concept'].str.contains('Volatility surface').any()
assert surface.groupby('expiry_days')['implied_vol'].count().min() > 1

## 4. Volatility premium versus variance premium

The lecture shorthand IV - RV is a volatility premium in vol points. For variance units, use IV^2 - RV^2.

In [ ]:
prem_ref = data['ots02_variance_premium_reference.csv']
prem = data['ots02_variance_premium_demo.csv']
print(prem_ref[['topic','correct_implementation','risk_control']].to_string(index=False))
print(prem[['implied_vol','realized_vol','vol_premium','variance_premium']].mean().round(5))
assert prem_ref['correct_implementation'].str.contains('IV^2', regex=False).any()
assert (prem['vol_premium'].mean() > 0)

## 5. Live trading controls

A volatility edge is tradable only after fills, costs, surface sanity, tail risk, and journaled regime review are included.

In [ ]:
controls = data['ots02_live_trading_controls.csv']
adverse = data['ots02_adverse_selection_cost_demo.csv']
print(controls.to_string(index=False))
print(adverse.to_string(index=False))
assert controls['control'].str.contains('Tail').any()
assert adverse['backtest_mean_all_signals'].iloc[0] > adverse['live_mean_filled_only'].iloc[0]